In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense

In [3]:
data = pd.read_csv('train.txt',sep=';')
data.columns = ['Text','Emotions']
print(data.head())

                                                Text Emotions
0  i can go from feeling so hopeless to so damned...  sadness
1   im grabbing a minute to post i feel greedy wrong    anger
2  i am ever feeling nostalgic about the fireplac...     love
3                               i am feeling grouchy    anger
4  ive been feeling a little burdened lately wasn...  sadness


In [13]:
texts = data['Text'].tolist()
labels = data['Emotions'].tolist()

#tokenize the text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

In [14]:
sequences = tokenizer.texts_to_sequences(texts)

max_length = max([len(seq) for seq in sequences])

padded_sequences = pad_sequences(sequences, maxlen=max_length)

In [15]:
padded_sequences

array([[   0,    0,    0, ...,    3,   21, 1254],
       [   0,    0,    0, ...,    2,  494,  437],
       [   0,    0,    0, ...,   29,    5, 3496],
       ...,
       [   0,    0,    0, ...,    3,  101, 1331],
       [   0,    0,    0, ...,  339,    8,   42],
       [   0,    0,    0, ...,   25, 3585,   12]],
      shape=(15999, 66), dtype=int32)

In [16]:
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

In [17]:
# one hot encode the labels
one_hot_labels = keras.utils.to_categorical(labels)

In [18]:
# training and testing dataset
X_train , X_test , y_train , y_test = train_test_split(padded_sequences , one_hot_labels , test_size = 0.2)

In [21]:
#define the model
model = Sequential()
model.add(Embedding(input_dim = len(tokenizer.word_index) + 1 ,
                   output_dim = 128 , input_length= max_length))
model.add(Flatten())
model.add(Dense(units = 128 , activation = 'relu'))
model.add(Dense(units = len(one_hot_labels[0]) , activation= 'softmax'))

model.compile(optimizer = 'adam' , loss = 'categorical_crossentropy' , metrics = ['accuracy'])
model.fit(X_train , y_train , epochs = 10 , batch_size = 32 , validation_data= (X_test , y_test))

C:\Users\Anushka\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.4774 - loss: 1.3445 - val_accuracy: 0.7172 - val_loss: 0.8462
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.8946 - loss: 0.3359 - val_accuracy: 0.8109 - val_loss: 0.5785
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9848 - loss: 0.0589 - val_accuracy: 0.8272 - val_loss: 0.5658
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.9941 - loss: 0.0273 - val_accuracy: 0.8241 - val_loss: 0.6126
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9954 - loss: 0.0199 - val_accuracy: 0.8250 - val_loss: 0.6395
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.9962 - loss: 0.0177 - val_accuracy: 0.8188 - val_loss: 0.6674
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9973 - loss: 0.0132 - val_accuracy: 0.8209 - val_loss: 0.7112
Epoch 8/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9966 - loss: 0.0138 - val_accu

In [24]:
input_text = "She didn't come today because she lost her dog yesterday!"

# Convert text to sequence
input_sequence = tokenizer.texts_to_sequences([input_text])

# Pad the sequence to the same length used during training
padded_input_sequence = pad_sequences(input_sequence, maxlen=max_length)

# Predict
prediction = model.predict(padded_input_sequence)

# Get predicted class index
predicted_index = np.argmax(prediction[0])

# Convert index back to original label
prediction_label = label_encoder.inverse_transform([predicted_index])

print(prediction_label[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
sadness


In [25]:
sample_texts = [
    "I absolutely loved this movie.",
    "The food was terrible and I hated it.",
    "Today is a beautiful day.",
    "I am feeling very sad today.",
    "She won the competition and was very happy.",
    "The customer service was disappointing.",
    "I have a meeting at 10 AM tomorrow.",
    "The weather is pleasant and sunny.",
    "He lost his wallet yesterday.",
    "This is the best product I have ever used."
]

for text in sample_texts:
    # Convert text to sequence
    input_sequence = tokenizer.texts_to_sequences([text])

    # Pad the sequence
    padded_input_sequence = pad_sequences(
        input_sequence,
        maxlen=max_length
    )

    # Predict
    prediction = model.predict(padded_input_sequence, verbose=0)

    # Get predicted class
    predicted_index = np.argmax(prediction[0])

    # Convert index to label
    predicted_label = label_encoder.inverse_transform([predicted_index])[0]

    print(f"Text: {text}")
    print(f"Predicted Label: {predicted_label}")
    print("-" * 50)

Text: I absolutely loved this movie.
Predicted Label: love
--------------------------------------------------
Text: The food was terrible and I hated it.
Predicted Label: sadness
--------------------------------------------------
Text: Today is a beautiful day.
Predicted Label: sadness
--------------------------------------------------
Text: I am feeling very sad today.
Predicted Label: sadness
--------------------------------------------------
Text: She won the competition and was very happy.
Predicted Label: joy
--------------------------------------------------
Text: The customer service was disappointing.
Predicted Label: joy
--------------------------------------------------
Text: I have a meeting at 10 AM tomorrow.
Predicted Label: sadness
--------------------------------------------------
Text: The weather is pleasant and sunny.
Predicted Label: joy
--------------------------------------------------
Text: He lost his wallet yesterday.
Predicted Label: sadness
-------------------